# 04. Clinical age acceleration and demo profiles

Combine the held-out age and mortality predictions to create this hackathon story:

**clinical age → age acceleration → downstream mortality association**.

This notebook is descriptive; it does not claim that changing an input causes the predicted age to change.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        root = candidate
        break
sys.path.insert(0, str(root))

from src.config import PROCESSED_DIR, DEMO_DIR, RESULTS_DIR, TARGET_AGE, TARGET_MORTALITY
from src.data_utils import load_json, save_json

age_pred = pd.read_csv(RESULTS_DIR / "age_test_predictions.csv")
mort_pred = pd.read_csv(RESULTS_DIR / "mortality_test_predictions.csv")
test = pd.read_csv(PROCESSED_DIR / "test_mortality.csv")
metadata = load_json(PROCESSED_DIR / "metadata.json")

analysis = age_pred.merge(mort_pred, on=["ROW_ID", "RIDAGEYR", "MORTSTAT"], how="inner", validate="one_to_one")
analysis["AGE_ACCELERATION"] = analysis["AGE_PRED_TABPFN"] - analysis["RIDAGEYR"]
print(analysis.head())
print("N:", len(analysis))

   ROW_ID  RIDAGEYR  AGE_PRED_TABPFN  AGE_ACCELERATION  MORTSTAT  \
0       7        38        36.886986         -1.113014       0.0   
1       8        85        80.316292         -4.683708       1.0   
2       9        23        25.506367          2.506367       0.0   
3      13        46        48.570786          2.570786       0.0   
4      19        21        46.813141         25.813141       0.0   

   MORTALITY_PROB_TABPFN  MORTALITY_PRED_TABPFN  
0               0.018578                      0  
1               0.974393                      1  
2               0.017190                      0  
3               0.144980                      0  
4               0.025279                      0  
N: 2057


In [2]:
# Descriptive comparison of mortality by age-acceleration quartile.
analysis["ACCELERATION_QUARTILE"] = pd.qcut(analysis["AGE_ACCELERATION"], q=4, labels=["Q1 youngest", "Q2", "Q3", "Q4 oldest"], duplicates="drop")
summary = analysis.groupby("ACCELERATION_QUARTILE", observed=False).agg(
    n=("MORTSTAT", "size"),
    mean_age_acceleration=("AGE_ACCELERATION", "mean"),
    mortality_rate=("MORTSTAT", "mean"),
    mean_mortality_probability=("MORTALITY_PROB_TABPFN", "mean"),
).reset_index()
summary.to_csv(RESULTS_DIR / "age_acceleration_mortality_summary.csv", index=False)
summary

,ACCELERATION_QUARTILE,n,mean_age_acceleration,mortality_rate,mean_mortality_probability
0,Q1 youngest,515,-9.513964,0.471845,0.450990
1,Q2,514,-1.801051,0.377432,0.353832
2,Q3,514,2.523164,0.235409,0.240155
3,Q4 oldest,514,10.013104,0.186770,0.194354


In [3]:
from scipy.stats import mannwhitneyu, spearmanr

rho, p_spearman = spearmanr(analysis["AGE_ACCELERATION"], analysis["MORTSTAT"], nan_policy="omit")
acc_dead = analysis.loc[analysis["MORTSTAT" == 1], "AGE_ACCELERATION"] if False else analysis.loc[analysis["MORTSTAT"] == 1, "AGE_ACCELERATION"]
acc_alive = analysis.loc[analysis["MORTSTAT"] == 0, "AGE_ACCELERATION"]
u_stat, p_mwu = mannwhitneyu(acc_dead, acc_alive, alternative="two-sided")

stats = {
    "spearman_rho_age_acceleration_vs_mortality": float(rho),
    "spearman_p": float(p_spearman),
    "mann_whitney_u": float(u_stat),
    "mann_whitney_p": float(p_mwu),
    "mean_acceleration_alive": float(acc_alive.mean()),
    "mean_acceleration_dead": float(acc_dead.mean()),
}
save_json(stats, RESULTS_DIR / "age_acceleration_stats.json")
stats

{'spearman_rho_age_acceleration_vs_mortality': -0.2527556684683661,
 'spearman_p': 2.400453638873282e-31,
 'mann_whitney_u': 315013.0,
 'mann_whitney_p': 2.0784061587137607e-30,
 'mean_acceleration_alive': 1.6275913730656684,
 'mean_acceleration_dead': -2.546331102330386}

In [4]:
# Choose two real held-out NHANES profiles with the SAME chronological age
# but very different clinical-age acceleration.
# This gives the app a visually compelling, non-causal comparison.
features = metadata["mortality_features"]
profile_base = test[["ROW_ID", "RIDAGEYR"] + features].copy()
profile_base = profile_base.merge(
    analysis[["ROW_ID", "AGE_ACCELERATION"]],
    on="ROW_ID",
    how="inner",
)

# Find the age with the largest within-age acceleration range, requiring >=2 people.
# Remove duplicate RIDAGEYR columns before grouping.
# This can happen when the feature list or the test frame already contains RIDAGEYR.
profile_base = profile_base.loc[:, ~profile_base.columns.duplicated()].copy()

by_age = (
    profile_base.groupby("RIDAGEYR")["AGE_ACCELERATION"]
    .agg(["count", "min", "max"])
)
by_age["range"] = by_age["max"] - by_age["min"]
valid_ages = by_age.loc[by_age["count"] >= 2]

if len(valid_ages) > 0:
    selected_age = valid_ages["range"].idxmax()
    band = profile_base.loc[profile_base["RIDAGEYR"] == selected_age].copy()
else:
    # Fallback: use the nearest pair to age 55 if no age has >=2 observations.
    profile_base["distance_to_55"] = (profile_base["RIDAGEYR"] - 55).abs()
    band = profile_base.sort_values("distance_to_55").head(20).copy()
    selected_age = int(band.iloc[0]["RIDAGEYR"])

patient_a = band.loc[band["AGE_ACCELERATION"].idxmin()].copy()
patient_b = band.loc[band["AGE_ACCELERATION"].idxmax()].copy()

demo_cols = ["profile_id"] + features
row_a = patient_a.reindex(["RIDAGEYR"] + features).to_dict()
row_b = patient_b.reindex(["RIDAGEYR"] + features).to_dict()
row_a["profile_id"] = "Profile A"
row_b["profile_id"] = "Profile B"

demo_profiles = pd.DataFrame([row_a, row_b])[demo_cols]
DEMO_DIR.mkdir(parents=True, exist_ok=True)
demo_profiles.to_csv(DEMO_DIR / "demo_profiles.csv", index=False)

print(f"Selected shared chronological age: {selected_age}")
print(demo_profiles[["profile_id", "RIDAGEYR"]])

Selected shared chronological age: 39
  profile_id  RIDAGEYR
0  Profile A      39.0
1  Profile B      39.0
